<a href="https://colab.research.google.com/github/ColumbiaPlus/GENAI_BizAnalytics/blob/main/Module2/RNN_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Recurrent Neural Network Example**
* Data: a series of numbers
* Learn: the next number in the series

**Important data note**
* RNNs are designed to predict the next value in a sequence of fixed size
* We will need to convert the input data into sequences


**Example: Arithmetic sequence**
* An arithmetic sequence with a difference of 2 is 1, 3, 5, 7, ......
* Input data to the RNN will consist of subsequence of size n. For n = 4:
** 1, 3, 5, 7 => 9
** 3, 5, 7, 9 => 11
** 5, 7, 9, 11 => 13
** etc.

## **Imports**

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN
from sklearn.model_selection import train_test_split

**Generate the data**
* An arithmetic sequence with difference = 3
* For practical execution purposes, we generate a sequence of 100,000 numbers

In [56]:
start = 1
diff = 3
n = 100000
progression = [start + diff*i for i in range (n)]
progression[999]
df = pd.DataFrame(progression)

In [57]:
len(df)

100000

In [58]:
df

,0
0,1
1,4
2,7
3,10
4,13
...,...
99995,299986
99996,299989
99997,299992
99998,299995


**Generate input sequences**
* We'll generate sequences of size k
* $ X_0 $ = [1,4,7,10]; $ y_0 $ = 13
* $ X_1 $ = [4,7,10,13]; $ y_0 $ = 16
* Etc.

In [59]:
def makeSequence(df, k):
  X, y =[], []
  for i in range(len(df)- k):
    d=i+k
    X.append(df.iloc[i:d,])
    y.append(df.iloc[d,])
  return np.array(X),np.array(y)

X,y = makeSequence(df,4)

In [60]:
X[:2]

array([[[ 1],
        [ 4],
        [ 7],
        [10]],

       [[ 4],
        [ 7],
        [10],
        [13]]])

In [61]:
y[:2]

array([[13],
       [16]])

In [62]:
#X has 99,996 samples; each sample is a sequence of 4 numbers; and each number is represented by 1 feature (its value)
X.shape

(99996, 4, 1)

**Input shape**
* RNNs require that the input shape be 3 dimensional
* (samples, timesteps, features)
* samples: the number of samples in the training data (length of X_train)
* timesteps: The number of lookback periods
** our data uses timestep of 4 because we're using 4 value sequences to predict the next value
** alternatively, we could say "use the value at t(i) to predict t(i+1) but also use the lagged values of t(i-1), t(i-2), t(i-3)
* features: the features that represent a token
* Note that this is exactly what we did with embedded vectors in module 1. In that example:
** samples: the number of sentences
** timesequence: the sequence of words (usually the length of the longest sentence)
** features: the embedding for each word in the sequence


In [63]:
#Understanding timesteps

#Assume our data is:
x_demo = np.array([[1,2],[1,3],[2,3],[5,6],[3,4]])
y_demo = np.array([3,4,5,11,7])

#I.e., y(0) = x(0,0) + x(0)(1)

#But, we want y(t) to depend on x(t) and x(t-1)
#We want to use two timesteps to determine y

look_back = 2 #This step plus the previous step

#Since we're looking at 2 timesteps, we're going to have one fewer data items
num_samples = x_demo.shape[0]-look_back + 1
num_features = 2 #The number of features at each timestep

#Create empty arrays for x and y reshaped
x_demo_reshaped = np.zeros((num_samples, look_back, num_features))
y_demo_reshaped = np.zeros((num_samples))
print(y_demo_reshaped)

#Iterate through the data creating x(t-1) and x(t) data for each y
for i in range(num_samples):
    print(i)
    y_position = i + look_back
    x_demo_reshaped[i] = x_demo[i:y_position]
    y_demo_reshaped[i] = y_demo[y_position-1]

x_demo_reshaped,y_demo_reshaped,x_demo_reshaped.shape

[0. 0. 0. 0.]
0
1
2
3


(array([[[1., 2.],
         [1., 3.]],
 
        [[1., 3.],
         [2., 3.]],
 
        [[2., 3.],
         [5., 6.]],
 
        [[5., 6.],
         [3., 4.]]]),
 array([ 4.,  5., 11.,  7.]),
 (4, 2, 2))

**Train and test samples**
* X is already in the correct shape
* We'll create a separate training and testing sample

In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)



In [25]:
X_train.shape

(69997, 4, 1)

In [27]:
X_train[:2]

array([[[259381],
        [259384],
        [259387],
        [259390]],

       [[245491],
        [245494],
        [245497],
        [245500]]])

In [28]:
y_train[:2]

array([[259393],
       [245503]])

**A Simple RNN model**
* keras provides us with the functionality we need to set up an RNN
* Our model consists of:
** One input layer of 4 timesteps and 1 feature. I.e., a two dimensional matrix of shape (4,1)
** An RNN layer with 32 units
** Each unit is a hidden layer
** Each timestep is passed through the hidden layer
** Though, theoretically, each timestep has its own hidden layer, in reality the same hidden layer is used
* We also have an additional normal hidden layer of 8 nodes that takes the output of the simple RNN and feeds it into the output layer (1 node)


In [68]:
# SimpleRNN model
model = Sequential()
model.add(SimpleRNN(units=32, input_shape=(4,1),activation="relu"))
model.add(Dense(8, activation="relu"))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='rmsprop')
model.summary()

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn_9 (SimpleRNN)    (None, 32)                1088      
                                                                 
 dense_18 (Dense)            (None, 8)                 264       
                                                                 
 dense_19 (Dense)            (None, 1)                 9         
                                                                 
Total params: 1361 (5.32 KB)
Trainable params: 1361 (5.32 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [67]:
X_train.shape

(69997, 4, 1)

**Running the model**
* RNNs are slow to train
* We'll run 10 epochs, 100 epochs, and 1000 epochs and compare the results

In [49]:
model.fit(X_train,y_train, epochs=10, batch_size=32, verbose=0)
trainPredict = model.predict(X_train)
testPredict= model.predict(X_test)
predicted=np.concatenate((trainPredict,testPredict),axis=0)
trainScore = model.evaluate(X_train, y_train, verbose=0)
testScore = model.evaluate(X_test,y_test,verbose=2)
print("Training score         :",trainScore)
print("Testing score          :",testScore)
print("First three predictions:",model.predict(X_test)[:3])
print("First three sequences  :", X_test[:3])

938/938 [==============================] - 2s 2ms/step
938/938 - 2s - loss: 480700.9062 - 2s/epoch - 2ms/step
Training score         : 485531.25
Testing score          : 480700.90625
938/938 [==============================] - 2s 2ms/step
First three predictions: [[101506.23]
 [ 73550.57]
 [127528.64]]
First three sequences  : [[[101905]
  [101908]
  [101911]
  [101914]]

 [[ 73837]
  [ 73840]
  [ 73843]
  [ 73846]]

 [[128032]
  [128035]
  [128038]
  [128041]]]


**Results after 10 epochs**
* Pretty bad!
* The first three predicted values should be:
** Actual:    101917, 73849, and 128044
** Predicted: 101506, 73551, and 127529
* The testing error is 480701
<pre>
2188/2188 [==============================] - 5s 2ms/step
938/938 [==============================] - 2s 2ms/step
938/938 - 2s - loss: 480700.9062 - 2s/epoch - 2ms/step
Training score         : 485531.25
Testing score          : 480700.90625
938/938 [==============================] - 2s 2ms/step
First three predictions:
 [[101506.23]
 [ 73550.57]
 [127528.64]]
First three sequences  :
[[[101905]
  [101908]
  [101911]
  [101914]]

 [[ 73837]
  [ 73840]
  [ 73843]
  [ 73846]]

 [[128032]
  [128035]
  [128038]
  [128041]]]
  </pre>

In [50]:
model.fit(X_train,y_train, epochs=100, batch_size=32, verbose=0)
trainPredict = model.predict(X_train)
testPredict= model.predict(X_test)
predicted=np.concatenate((trainPredict,testPredict),axis=0)
trainScore = model.evaluate(X_train, y_train, verbose=0)
testScore = model.evaluate(X_test,y_test,verbose=2)
print("Training score         :",trainScore)
print("Testing score          :",testScore)
print("First three predictions:",model.predict(X_test)[:3])
print("First three sequences  :", X_test[:3])

938/938 [==============================] - 2s 3ms/step
938/938 - 3s - loss: 172503.8594 - 3s/epoch - 4ms/step
Training score         : 174240.46875
Testing score          : 172503.859375
938/938 [==============================] - 3s 3ms/step
First three predictions: [[101671.38]
 [ 73670.8 ]
 [127735.6 ]]
First three sequences  : [[[101905]
  [101908]
  [101911]
  [101914]]

 [[ 73837]
  [ 73840]
  [ 73843]
  [ 73846]]

 [[128032]
  [128035]
  [128038]
  [128041]]]


**Results after 100 epochs**
* Better!
* The first three predicted values should be:
** Actual:         101917, 73849, and 128044
** Predicted(100): 101671, 73671, and 127736
** Predicted(10):  101506, 73551, and 127529
* The testing error has dropped to 172504 from 480701

<pre>
2188/2188 [==============================] - 7s 3ms/step
938/938 [==============================] - 2s 3ms/step
938/938 - 3s - loss: 172503.8594 - 3s/epoch - 4ms/step
Training score         : 174240.46875
Testing score          : 172503.859375
938/938 [==============================] - 3s 3ms/step
First three predictions: [[101671.38]
 [ 73670.8 ]
 [127735.6 ]]
First three sequences  : [[[101905]
  [101908]
  [101911]
  [101914]]

 [[ 73837]
  [ 73840]
  [ 73843]
  [ 73846]]

 [[128032]
  [128035]
  [128038]
  [128041]]]
  </pre>

In [51]:
model.fit(X_train,y_train, epochs=1000, batch_size=32, verbose=0)
trainPredict = model.predict(X_train)
testPredict= model.predict(X_test)
predicted=np.concatenate((trainPredict,testPredict),axis=0)
trainScore = model.evaluate(X_train, y_train, verbose=0)
testScore = model.evaluate(X_test,y_test,verbose=2)
print("Training score         :",trainScore)
print("Testing score          :",testScore)
print("First three predictions:",model.predict(X_test)[:3])
print("First three sequences  :", X_test[:3])

938/938 [==============================] - 2s 2ms/step
938/938 - 2s - loss: 102.8322 - 2s/epoch - 2ms/step
Training score         : 103.92191314697266
Testing score          : 102.8321762084961
938/938 [==============================] - 2s 2ms/step
First three predictions: [[101911.28 ]
 [ 73845.016]
 [128036.67 ]]
First three sequences  : [[[101905]
  [101908]
  [101911]
  [101914]]

 [[ 73837]
  [ 73840]
  [ 73843]
  [ 73846]]

 [[128032]
  [128035]
  [128038]
  [128041]]]


**Results after 100 epochs**
* Much much better!
* The first three predicted values should be:
** Actual:          101917, 73849, and 128044
** Predicted(1000): 101911, 73845, and 128037
** Predicted(100):  101671, 73671, and 127736
** Predicted(10):   101506, 73551, and 127529
* The testing error has dropped to to 103 from 172504 at 100 epochs and 480701 at 10 epochs
* We could get closer to learning the sequence by
** Training for more epochs
** Using longer sequences
** A mix of both


<pre>
2188/2188 [==============================] - 5s 2ms/step
938/938 [==============================] - 2s 2ms/step
938/938 - 2s - loss: 102.8322 - 2s/epoch - 2ms/step
Training score         : 103.92191314697266
Testing score          : 102.8321762084961
938/938 [==============================] - 2s 2ms/step
First three predictions: [[101911.28 ]
 [ 73845.016]
 [128036.67 ]]
First three sequences  : [[[101905]
  [101908]
  [101911]
  [101914]]

 [[ 73837]
  [ 73840]
  [ 73843]
  [ 73846]]

 [[128032]
  [128035]
  [128038]
  [128041]]]
  </pre>